# V13 — does unfreezing the backbone close the archival-to-field gap?

Every result in this project trains a head on cached activations from a VGG19
frozen on ImageNet weights. That is what Sun et al. (2022) do and what this
pipeline inherited, and it has never been tested here.

The measurement that makes it worth testing: in the model's own feature space
the library training clips sit about **50** Mahalanobis units from their class
centre and field audio sits about **1860** — a factor of thirty-seven. ImageNet
weights have no reason to place a close-range library recording of a roar near a
hundred-metre field recording of the same roar, and no amount of re-cutting or
cleaning the training material can move features the head does not control.
Three attempts to fix the Colobus class by changing how it is cut have now
failed; this is the first that can move the features themselves.

**Two runs, differing in one thing.** The frozen arm is run here too rather than
compared against the laptop numbers, because those predate early stopping. Same
folds, same patience, same data — only `--unfreeze` differs.

**What this notebook cannot do.** Detection needs the 444 GB of field recordings,
which are not here. It trains, reports leave-one-station-out precision, and saves
the weights back to Drive; detection and listening happen on the laptop. That
matters because validation accuracy has been 95–96 % throughout this project and
has never once predicted field behaviour.

In [ ]:
# Runtime -> Change runtime type -> T4 GPU, before running anything else.
!nvidia-smi -L
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, 'GPUs:', gpus)
assert gpus, 'No GPU. Fine-tuning the base on CPU is ~8x slower per epoch than the cached path and not worth starting.'

## 1. Code and data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/primates-sound-detection/colab_upload'
!ls -la $DRIVE

In [ ]:
# Clone rather than copy, so what runs here is a commit rather than a snapshot
# that has quietly drifted from the laptop.
!rm -rf /content/repo
!git clone -q https://github.com/Mo119m/primates-sound-detection /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
# Off Drive onto local disk. Drive is a network mount and the image pack is read
# once per epoch; leaving it there makes the run I/O-bound on a machine rented
# for its GPU.
import os, shutil, time
os.makedirs('/content/data', exist_ok=True)
for f in ['v13_images.npy', 'v13_index.csv', 'manifest.csv']:
    dst = f'/content/data/{f}'
    if not os.path.exists(dst):
        t0 = time.time()
        shutil.copy(f'{DRIVE}/{f}', dst)
        print(f'{f}: {os.path.getsize(dst)/1e9:.2f} GB in {time.time()-t0:.0f}s')
!ls -la /content/data

In [ ]:
# The artifacts have to describe one dataset before anything trains on them.
!python scripts/check_v13_artifacts.py \
  --manifest /content/data/manifest.csv \
  --index /content/data/v13_index.csv \
  --images /content/data/v13_images.npy \
  --cache /content/data/v13_features.npy || echo 'cache absent is expected on the first run'

## 2. Frozen arm — the control

Three folds spanning the difficulty range as measured by the deployed model:
IPA20ST at 0.935, IPA13ST at 0.773, IPA4ST at 0.695. The feature cache is a
deterministic function of the image pack and the frozen base, so it is built
once here and thrown away when the base stops being frozen.

In [ ]:
FOLDS = 'IPA20ST,IPA13ST,IPA4ST'
ARGS = ('--manifest /content/data/manifest.csv '
        '--index /content/data/v13_index.csv '
        '--images /content/data/v13_images.npy '
        '--cache /content/data/v13_features.npy')

!python scripts/train_v13_loso.py --prepare-cache-only --overwrite {ARGS} \
  --out /content/out_frozen.csv --run-metadata /content/cache.run.json

In [ ]:
!python scripts/train_v13_loso.py --folds {FOLDS} --epochs 15 --patience 3 \
  --overwrite {ARGS} --out /content/out_frozen.csv

## 3. Fine-tuned arm

`--unfreeze 2` releases blocks 5 and 4 after the head has been fitted, at 1e-5
against the head's 1e-4. The order is not a detail: a randomly initialised head
backpropagating into pretrained convolutions destroys them, which is the failure
mode fine-tuning is known for.

This reads the image pack, not the cache — the cache is the frozen base's output
and reusing it would carry the assumption under test.

In [ ]:
!python scripts/train_v13_loso.py --folds {FOLDS} --epochs 15 --patience 3 \
  --unfreeze 2 --finetune-epochs 5 --finetune-lr 1e-5 \
  --overwrite {ARGS} --out /content/out_finetune.csv \
  --head-dir /content/heads_finetune

## 4. The comparison

In [ ]:
import pandas as pd
froz = pd.read_csv('/content/out_frozen.csv')
fine = pd.read_csv('/content/out_finetune.csv')
cols = ['gated_v12_precision', 'gated_loso_precision',
        'gated_loso_calls_retained', 'gated_loso_fps_removed']
cmp = pd.DataFrame({'frozen': froz[cols].mean(), 'fine-tuned': fine[cols].mean()})
cmp['delta'] = cmp['fine-tuned'] - cmp['frozen']
print(cmp.round(4).to_string())
print()
print('per station:')
print(froz[['station', 'gated_loso_precision']].merge(
    fine[['station', 'gated_loso_precision']], on='station',
    suffixes=('_frozen', '_finetuned')).round(4).to_string(index=False))

**Reading this honestly.** Three folds is a screen, not a result: it tells you
whether the direction is worth sixteen folds, and a difference smaller than the
spread between the three stations is not a difference. The number that decides
whether this method is usable is still precision on raw audio, checked by ear —
51/55 at 0.95 and above for the frozen model — and that needs the field
recordings, which are on the laptop.

## 5. Back to Drive

In [ ]:
OUT = '/content/drive/MyDrive/primates-sound-detection/colab_results'
!mkdir -p $OUT
!cp /content/out_frozen.csv /content/out_finetune.csv $OUT/
!cp -r /content/heads_finetune $OUT/ 2>/dev/null || true
!ls -la $OUT
print('\nThe head weights are the deliverable: assemble_fold_model.py welds them to')
print('the base on the laptop, and detection runs there where the audio is.')